# Stage 1 · Minimal RAG

Goal: answer questions about **a single document**, without a vector database.

Blocks:
1. Read the PDF (page by page)
2. Split it into chunks, keeping track of the source page
3. Compute embeddings *(next session)*
4. Retrieve and answer *(next session)*

In [ ]:
from pathlib import Path
import pypdf

## Block 1 · Read the PDF

Unlike the course's `read_pdf` (which joins all the text into one string), we keep **each page separately**. Later, the assistant will be able to say *"this comes from page 4"*.

In [ ]:
def read_pdf(path):
    """Return a list of dicts, one per page, with its page number and text."""
    reader = pypdf.PdfReader(path)
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):  # start=1 so the first page is 1, not 0
        text = page.extract_text() or ""                          # some pages (e.g. image only) have no text
        pages.append({"page": page_number, "text": text})
    return pages

In [ ]:
PDF_PATH = Path("data/rag_papers/lec1_slides.pdf")   # change this path to your document

pages = read_pdf(PDF_PATH)
print("Pages:", len(pages))
print("Total characters:", sum(len(p["text"]) for p in pages))
print("\n--- Page 4 (first 500 characters) ---")
print(pages[3]["text"][:500])

## Block 2 · Split into chunks

Same idea as the course's `naive_chunks`: windows of `chunk_size` characters that move forward by `chunk_size - overlap`.

The difference: we split **page by page** and attach a *label* to each chunk (document, page, chunk number), like writing on the back of each index card where it came from.

In [ ]:
def split_into_chunks(pages, document_name, chunk_size=1000, overlap=200):
    if not 0 <= overlap < chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")

    step = chunk_size - overlap
    chunks = []
    for p in pages:
        text = p["text"]
        for start in range(0, len(text), step):
            piece = text[start:start + chunk_size]
            if len(piece.strip()) < 50:          # skip almost empty leftovers at the end of a page
                continue
            chunks.append({
                "document": document_name,
                "page": p["page"],
                "chunk_num": len(chunks),
                "text": piece,
            })
            if start + chunk_size >= len(text):  # reached the end of the page
                break
    return chunks

In [ ]:
chunks = split_into_chunks(pages, document_name=PDF_PATH.stem)

print("Number of chunks:", len(chunks))
example = chunks[5]
print(f"\n--- Chunk {example['chunk_num']} | {example['document']} | page {example['page']} ---")
print(example["text"])

### Things to check yourself

- Is there text repeated on every page (headers, copyright notices)?
- Are there chunks starting with a word cut in half?
- Try `chunk_size=300` and `chunk_size=3000`: how many chunks do you get?